In [20]:
import os
from glob import glob
import warnings
import pandas as pd
import numpy as np
from tqdm import tqdm_notebook as tqdm
import xgboost as xgb

from settings import ROOT
from settings import DATA_PATH
from settings import TRAIN_DATA_PATH
from settings import TEST_DATA_PATH
from settings import RESULT_PATH

warnings.filterwarnings("ignore")

# 1.训练数据

## 成型机状态数据

In [2]:
spc_df = pd.read_csv(os.path.join(TRAIN_DATA_PATH, "data_spc.csv"))
spc_df.head()

,Id,EL_CYC_TIME,EL_IV_END_TIME,EL_PLST_TIME,EL_LAST_COOL_TIME,EL_MC_TIME,EL_MO_TIME,EL_EF_TIME,EL_EB_TIME,EL_IDLE_TIME,...,EL_PP_STG_POS_4,EL_CLAMP_PRESS,EL_MAX_INJ_PRESS,EL_IV_END_PRESS,EL_CF_END_PRESS,EL_PLST_BP_0,EL_PLST_BP_1,EL_PLST_BP_2,EL_PLST_BP_3,EL_PLST_BP_4
0,33722,37.376,NaN,6.252,5.748,3.101,1.969,1.057,1.915,0.0,...,10.0,131.310486,75.008865,45.194874,5.0,5.0,5.000546,5.0,5.0,5.0
1,34803,37.362,NaN,6.762,5.238,3.093,1.965,1.054,1.913,0.0,...,10.0,129.210010,75.701614,45.527172,5.0,5.0,5.000000,5.0,5.0,5.0
2,36621,37.376,NaN,6.254,5.746,3.106,1.968,1.053,1.910,0.0,...,10.0,129.424430,75.730300,45.562855,5.0,5.0,5.000121,5.0,5.0,5.0
3,36622,37.372,NaN,6.605,5.395,3.098,1.968,1.052,1.911,0.0,...,10.0,129.654945,73.227135,45.607864,5.0,5.0,5.000000,5.0,5.0,5.0
4,36623,37.360,NaN,6.207,5.793,3.090,1.965,1.055,1.912,0.0,...,10.0,129.547615,71.181970,46.110380,5.0,5.0,5.000081,5.0,5.0,5.0


In [3]:
spc_df.shape

(16600, 86)

## Size.csv 数据

In [4]:
size_df = pd.read_csv(os.path.join(TRAIN_DATA_PATH, "Size.csv"))
size_df.head()

,size1,size2,size3,Id
0,299.998,199.995,199.967,33722
1,299.999,199.938,199.956,34803
2,299.958,199.916,199.947,36621
3,299.974,199.930,199.961,36622
4,299.959,199.915,199.949,36623


In [5]:
size_df.shape

(16600, 4)

## 传感器高频数据

In [6]:
sensor_df = pd.read_csv(os.path.join(TRAIN_DATA_PATH, "sensor_train.csv"))
sensor_df.head()

,Id,SampleTime,Phase_max,Phase_min,Phase_max-min,Phase_std,Phase_skew,Phase_mean,Sensor1_max,Sensor1_min,...,MouldFlow3_max-min,MouldFlow3_std,MouldFlow3_skew,MouldFlow3_mean,SP_max,SP_min,SP_max-min,SP_std,SP_skew,SP_mean
0,37933,37.34,18.0,1.0,17.0,2.925510,1.539405,5.782330,42.614,5.395,...,0.0,0.0,0,5.0,104.736,32.654,72.082,31.406682,0.221530,65.951627
1,144415,41.21,18.0,1.0,17.0,2.946129,1.163427,6.182086,54.747,5.299,...,0.0,0.0,0,5.0,104.659,34.756,69.903,30.248818,-0.131981,73.053732
2,148371,41.31,18.0,1.0,17.0,2.956767,1.138020,6.204826,55.837,5.314,...,0.0,0.0,0,5.0,104.637,35.310,69.327,30.035895,-0.142125,73.512720
3,46444,37.29,18.0,1.0,17.0,2.898521,1.630998,5.720692,44.728,5.311,...,0.0,0.0,0,5.0,104.559,33.617,70.942,29.930991,0.306738,65.540175
4,49633,37.34,18.0,1.0,17.0,2.901625,1.623669,5.725992,44.115,5.325,...,0.0,0.0,0,5.0,104.575,33.444,71.131,29.960745,0.302199,65.633237


In [7]:
sensor_df.shape

(16600, 140)

## 所有的训练数据

In [8]:
all_data = pd.merge(spc_df, size_df, on = "Id")
all_data = all_data.dropna(how = "all", axis = 1)
all_data.isnull().sum().sum()
all_data = pd.merge(all_data, sensor_df, on = "Id")
all_data.head()

,Id,EL_CYC_TIME,EL_PLST_TIME,EL_LAST_COOL_TIME,EL_MC_TIME,EL_MO_TIME,EL_EF_TIME,EL_EB_TIME,EL_IDLE_TIME,EL_CORE_IN_TIME_0,...,MouldFlow3_max-min,MouldFlow3_std,MouldFlow3_skew,MouldFlow3_mean,SP_max,SP_min,SP_max-min,SP_std,SP_skew,SP_mean
0,33722,37.376,6.252,5.748,3.101,1.969,1.057,1.915,0.0,1.1,...,0.0,0.0,0,5.0,104.646,31.691,72.955,31.735239,0.229333,65.295029
1,34803,37.362,6.762,5.238,3.093,1.965,1.054,1.913,0.0,1.1,...,0.0,0.0,0,5.0,104.515,31.215,73.300,31.702571,0.229004,64.859819
2,36621,37.376,6.254,5.746,3.106,1.968,1.053,1.910,0.0,1.1,...,0.0,0.0,0,5.0,104.597,31.514,73.083,31.726763,0.227112,65.313014
3,36622,37.372,6.605,5.395,3.098,1.968,1.052,1.911,0.0,1.1,...,0.0,0.0,0,5.0,104.591,31.706,72.885,31.523484,0.231116,65.176727
4,36623,37.360,6.207,5.793,3.090,1.965,1.055,1.912,0.0,1.1,...,0.0,0.0,0,5.0,104.581,32.735,71.846,31.177252,0.222591,66.066778


# 2.测试数据

## 成型机状态数据

In [9]:
spc_test = pd.read_csv(os.path.join(TEST_DATA_PATH, "data_spc.csv"))
spc_test.head()

,Id,EL_CYC_TIME,EL_IV_END_TIME,EL_PLST_TIME,EL_LAST_COOL_TIME,EL_MC_TIME,EL_MO_TIME,EL_EF_TIME,EL_EB_TIME,EL_IDLE_TIME,...,EL_PP_STG_POS_4,EL_CLAMP_PRESS,EL_MAX_INJ_PRESS,EL_IV_END_PRESS,EL_CF_END_PRESS,EL_PLST_BP_0,EL_PLST_BP_1,EL_PLST_BP_2,EL_PLST_BP_3,EL_PLST_BP_4
0,56689,37.627,NaN,7.107,4.893,3.102,1.970,1.052,1.909,0.0,...,10.0,129.243890,72.12393,47.294640,5.0,11.530634,11.743819,5.0,5.0,5.0
1,56691,37.612,NaN,7.064,4.936,3.089,1.967,1.053,1.909,0.0,...,10.0,129.761270,75.42702,47.171050,5.0,11.519929,11.529081,5.0,5.0,5.0
2,56697,37.614,NaN,6.963,5.037,3.096,1.964,1.049,1.907,0.0,...,10.0,129.806470,75.55316,47.063442,5.0,11.505568,11.529027,5.0,5.0,5.0
3,56698,37.622,NaN,7.028,4.972,3.097,1.962,1.057,1.911,0.0,...,10.0,129.581240,71.36458,47.417645,5.0,11.729644,11.499857,5.0,5.0,5.0
4,56701,37.620,NaN,7.108,4.892,3.096,1.963,1.053,1.909,0.0,...,10.0,129.786026,72.10709,46.655704,5.0,11.566742,11.370344,5.0,5.0,5.0


## 传感器高频数据

In [10]:
sensor_test = pd.read_csv(os.path.join(TEST_DATA_PATH, "sensor_test.csv"))
sensor_test.head()

,Id,SampleTime,Phase_max,Phase_min,Phase_max-min,Phase_std,Phase_skew,Phase_mean,Sensor1_max,Sensor1_min,...,MouldFlow3_max-min,MouldFlow3_std,MouldFlow3_skew,MouldFlow3_mean,SP_max,SP_min,SP_max-min,SP_std,SP_skew,SP_mean
0,163036,41.21,18,1,17,2.951782,1.139911,6.199764,51.538,5.339,...,0.0,0.0,0,5.0,104.655,35.150,69.505,30.100019,-0.132067,73.447635
1,167656,41.01,18,1,17,2.948494,1.138746,6.223272,50.007,5.357,...,0.0,0.0,0,5.0,104.538,35.602,68.936,29.922135,-0.150501,73.588325
2,170291,41.10,18,1,17,2.928021,1.208387,6.163325,53.574,5.301,...,0.0,0.0,0,5.0,104.647,35.334,69.313,29.802176,-0.129743,73.101010
3,83700,39.81,18,1,17,3.000619,1.464738,5.826355,56.274,5.292,...,0.0,0.0,0,5.0,104.652,32.815,71.837,30.282986,0.220271,66.592338
4,151941,41.26,18,1,17,2.957477,1.128127,6.211425,52.652,5.298,...,0.0,0.0,0,5.0,104.614,35.423,69.191,29.900024,-0.143572,73.723500


## 所有测试数据

In [11]:
all_test = pd.merge(spc_test, sensor_test, on = "Id")
Id = all_test["Id"].to_list()
all_test = all_test.drop(["Id", "remark", "spcTime"], axis = 1)
all_test = all_test.dropna(how = "all", axis=1)
all_test.head()

,EL_CYC_TIME,EL_PLST_TIME,EL_LAST_COOL_TIME,EL_MC_TIME,EL_MO_TIME,EL_EF_TIME,EL_EB_TIME,EL_IDLE_TIME,EL_CORE_IN_TIME_0,EL_CORE_IN_TIME_1,...,MouldFlow3_max-min,MouldFlow3_std,MouldFlow3_skew,MouldFlow3_mean,SP_max,SP_min,SP_max-min,SP_std,SP_skew,SP_mean
0,37.627,7.107,4.893,3.102,1.970,1.052,1.909,0.0,1.1,0.0,...,0.0,0.0,0,5.0,104.585,33.436,71.149,29.717039,0.296648,65.771777
1,37.612,7.064,4.936,3.089,1.967,1.053,1.909,0.0,1.1,0.0,...,0.0,0.0,0,5.0,104.547,33.188,71.359,29.781750,0.291345,65.735227
2,37.614,6.963,5.037,3.096,1.964,1.049,1.907,0.0,1.1,0.0,...,0.0,0.0,0,5.0,104.711,33.314,71.397,29.869805,0.281757,65.832506
3,37.622,7.028,4.972,3.097,1.962,1.057,1.911,0.0,1.1,0.0,...,0.0,0.0,0,5.0,104.494,33.107,71.387,29.853620,0.290798,65.688890
4,37.620,7.108,4.892,3.096,1.963,1.053,1.909,0.0,1.1,0.0,...,0.0,0.0,0,5.0,104.687,33.036,71.651,29.849679,0.296606,65.614154


In [12]:
all_test.shape

(3953, 215)

# 3.训练数据分割

In [13]:
Y = all_data[["size1", "size2", "size3"]]
X = all_data.drop(["size1", "size2", "size3", "Id", "remark", "spcTime"], axis = 1)

In [14]:
# from sklearn.model_selection import train_test_split
# X_train,X_val, y_train, y_val = train_test_split(X,Y,test_size=0.2, random_state=2020)

# 4.模型训练

In [15]:
def xgb_model(X_train, y_train, X_test, xgb_params, num_boost_round):
    d_train = xgb.DMatrix(X_train,y_train)
    d_test = xgb.DMatrix(X_test)
    res = xgb.cv(
        xgb_params, 
        d_train, 
        num_boost_round = num_boost_round, 
        nfold = 5, 
        seed = 2018, 
        stratified = False,
        early_stopping_rounds = 100, 
        verbose_eval = 100,
        show_stdv = True
    )
    gbdt = xgb.train(xgb_params, d_train, res.shape[0])
    
    return gbdt.predict(d_test)

xgb_params = {
    'seed': 2018,
    'colsample_bytree': 0.7,
    'subsample': 0.7,
    'learning_rate': 0.15,
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'num_parallel_tree': 1,
    'min_child_weight': 1,
    'eval_metric': 'rmse',
}
num_boost_round = 5000
y_size1 = xgb_model(X, Y["size1"], all_test, xgb_params, num_boost_round)
y_size2 = xgb_model(X, Y["size2"], all_test, xgb_params, num_boost_round)
y_size3 = xgb_model(X, Y["size3"], all_test, xgb_params, num_boost_round)

[0]	train-rmse:254.63553+0.00020	test-rmse:254.63553+0.00109
[100]	train-rmse:0.01646+0.00035	test-rmse:0.02091+0.00154
[200]	train-rmse:0.01137+0.00027	test-rmse:0.02100+0.00156
[0]	train-rmse:169.58286+0.00015	test-rmse:169.58286+0.00043
[100]	train-rmse:0.01560+0.00010	test-rmse:0.02025+0.00026
[200]	train-rmse:0.01087+0.00004	test-rmse:0.02035+0.00032
[0]	train-rmse:169.59333+0.00013	test-rmse:169.59333+0.00073
[100]	train-rmse:0.01428+0.00008	test-rmse:0.01795+0.00038
[200]	train-rmse:0.00998+0.00006	test-rmse:0.01801+0.00033


In [21]:
submit = pd.DataFrame({
    "ID": Id,
    "size1": y_size1,
    "size2": y_size2,
    "size3": y_size3
})
size1_max = 300.15
size1_min = 299.85
size2_max = 200.075
size2_min = 199.925
size3_max = 200.075
size3_min = 199.925
submit["size1"][submit["size1"] > size1_max] = size1_max
submit["size1"][submit["size1"] < size1_min] = size1_min
submit["size2"][submit["size2"] > size2_max] = size2_max
submit["size2"][submit["size2"] < size2_min] = size2_min
submit["size3"][submit["size3"] > size3_max] = size3_max
submit["size3"][submit["size3"] < size3_min] = size3_min

In [22]:
submit.to_csv(os.path.join(RESULT_PATH, "submit_xgb_seperate_limit.csv"),index = None)